# UDnet-U (งานใหม่) บน Colab

<a href="https://colab.research.google.com/github/nantapongpa-a11y/UDnet/blob/UDnet-Colab/UDnetU_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

UDnet ของ repo + องค์ประกอบใหม่จาก `UDnet_U_demo.ipynb` (โค้ดเดิมไม่แก้ อยู่ใน `model_utils/udnet_u.py`) เทรนด้วย `TrainU.py`
ใช้ Drive โฟลเดอร์และ split เดียวกับ `UDnet_Baseline_Colab.ipynb` — **ชุดทดสอบเดียวกัน ตารางผลรวมกัน**

| § | ทำอะไร | เวลา (T4) |
|---|---|---|
| 1 | ตั้งค่า (วิธี + hyperparameter อยู่ที่นี่ที่เดียว) | < 1 นาที |
| 2 | clone repo + ข้อมูล (UIEB เป็นโดเมนใต้น้ำ, EUVP เป็นโดเมนภาพสะอาด) | 3–10 นาทีครั้งแรก |
| 3 | smoke test กับ UDnet จริงบนข้อมูลจริง | ~1 นาที |
| 4 | **ด่านตัดสินใจ:** ขอบบนของ phase lock แต่ละแบบบน UIEB จริง | ~1 นาที |
| 5 | เทรน `TrainU.py` (ต่อเองถ้า runtime หลุด) | หลายชั่วโมง |
| 6 | ประเมิน + ตารางรวมกับเส้นฐาน | ~2 นาที |
| 6.1 | (ทางเลือก) วัด checkpoint ระหว่างทาง — เทรนสั้นกว่านี้พอไหม | ~4 นาที |
| 7 | กราฟ loss (L_m, L_s แยก) และภาพเปรียบเทียบ | < 1 นาที |

**ตั้งค่าตามแผนในสไลด์**
- ขั้นที่ 1: `STAGE=1`, `LAM_ADV` เล็ก, `LAM_E=1` (ยังไม่ตัด L_e), `LOCK="none"`, `LAM_PHYS=0`
- ขั้นที่ 2: `STAGE=2`, ตั้ง `WARMUP_ITERS` (เทรนแบบขั้น 1 ก่อน) และ `LE_ANNEAL` (ค่อยๆ ลด L_e), เปิด `LOCK` / `LAM_PHYS` ตามผล §4
- ablation: ปิดทีละตัวด้วย `LAM_ADV=0`, `LAM_PHYS=0`, `LOCK="none"` — ทุก run แยกโฟลเดอร์ตามชื่อที่บอกค่าที่ตั้ง

**physical loss มีสองแบบ (`PHYS_MODE`)**
- `"dcp"` — แบบใน demo: t, B∞ จาก underwater DCP (§4 วัดแล้วว่าเป้าของมันแย่กว่าภาพดิบ)
- `"learned"` — physics prior ของ Cheng et al., *Neurocomputing* 621 (2025): B = เบลอเกาส์เซียนของ input, t จากเครือข่ายเล็ก (UIIE-Subnet),
  และ t′, B′ จากภาพที่ปรับแล้ว (UDE-Subnet) พร้อม physical map consistency — ใช้เฉพาะตอนเทรน ไม่มี CycleGAN, inference ยังเท่า UDnet

**โดเมนภาพสะอาด (Y):** โฟลเดอร์ Google Drive ทางการของ EUVP ตอนนี้ขึ้น 404
จึงใช้สำเนาสาธารณะบน Hugging Face ([Ken1053/EUVP](https://huggingface.co/datasets/Ken1053/EUVP), ไม่มีรหัส, 189 MB)
ซึ่งเป็นชุด *Underwater Dark* ส่วน train ของ EUVP 5,550 คู่ — ใช้เฉพาะ**ฝั่งภาพคุณภาพดี**และไม่จับคู่
(ตรวจแล้วว่าคอลัมน์ `edited_image` คือภาพดี: สีสมดุล ส่วน `input_image` แดงต่ำอมฟ้าเขียว)
ในวิทยานิพนธ์ให้อ้าง EUVP (Islam et al., 2020) และระบุว่าใช้สำเนานี้ · ไม่มีภาพจาก UIEB ปน จึงไม่รั่วไปชุดทดสอบ

**เรื่องรหัสไฟล์ (ตรวจแล้ว):** UIEB บน Google Drive เป็น `.rar` ที่ **เข้ารหัส AES** รหัสเป็นของที่ผู้สร้างชุดข้อมูลเผยแพร่เองบน
[หน้าโครงการ UIEB](https://li-chongyi.github.io/proj_benchmark.html) (ใช้เพื่องานวิชาการที่ไม่ใช่เชิงพาณิชย์) — ใส่ให้แล้วใน `colab_utils.py`
ถ้าแตกไม่ได้หรือจำนวนภาพไม่ครบ notebook จะหยุดพร้อมบอกสาเหตุ · weight ผู้แต่ง `UDnet.pth` ไม่มีรหัส

ถ้าดาวน์โหลดขึ้น **quota exceeded / too many users**: เปิดลิงก์ที่พิมพ์ออกมา ดาวน์โหลดเอง แล้ววางไว้ใน `UDnet_work/downloads/` ด้วยชื่อเดิม แล้วรันเซลล์ใหม่

## §1 ตั้งค่า

In [ ]:
REPO_URL   = "https://github.com/nantapongpa-a11y/UDnet.git"
BRANCH     = "UDnet-Colab"
SEED       = 42      # seed ของ split — ต้องเท่ากับ notebook เส้นฐาน
TRAIN_SEED = 123     # เปลี่ยนตอนทำ 3 seed
N_SUBSET   = 800     # ภาพใต้น้ำ 800 / 400 / 200 (ชุดย่อยซ้อนกัน เหมือน notebook เส้นฐาน)
N_CLEAN    = None    # จำนวนภาพสะอาด; None = เท่ากับ N_SUBSET (อัตราส่วน 1:1 ตามแผน)
BATCH      = 6
ITERS      = 40000   # จำนวน iteration ล็อกเท่ากันทุก run — รอบแรกลอง 200 เพื่อเช็ก pipeline

# ---- วิธี: แก้ตรงนี้ที่เดียว ----
STAGE        = 1
LAM_E        = 1.0     # MSE + VGG เทียบ pseudo-label (L_e ของ UDnet)
BETA_KL      = 1.0     # KL (L_m + L_s) ของ UDnet
LAM_ADV      = 0.01    # adversarial บนโดเมนภาพสะอาด (0 = ปิด)
ADV_WARMUP   = 2000    # ไต่ λ_adv จาก 0 ในช่วงแรก
LAM_PHYS     = 0.0     # physical loss (0 = ปิด)
PHYS_MODE    = "dcp"   # "dcp" (demo) | "learned" (physics prior ของ Cheng et al. 2025)
GB_SIGMA     = 0.1     # learned: ขนาดเบลอเกาส์เซียนของ B เป็นสัดส่วนของขนาดภาพ (เปเปอร์ไม่ได้ระบุ)
LOCK         = "none"  # none | full | low | high — ตัดสินจากผล §4
LOCK_RADIUS  = 0.5
WARMUP_ITERS = 0       # ขั้น 2: เทรนแบบขั้น 1 ก่อนกี่ iteration
LE_ANNEAL    = 0       # ขั้น 2: ลด L_e จนเป็น 0 ภายในกี่ iteration หลัง warmup
INIT         = ""      # "" = เทรนจากศูนย์ (เทียบกับเส้นฐานที่เทรนจากศูนย์) | "author" = เริ่มจาก weight ผู้แต่ง
USE_DRIVE    = True

RUN_NAME = (f"udnetu_s{STAGE}_n{N_SUBSET}_adv{LAM_ADV}_phys{LAM_PHYS}" + (PHYS_MODE if LAM_PHYS else "")
            + f"_lock{LOCK}"
            + (f"{LOCK_RADIUS}" if LOCK in ("low", "high") else "")
            + (f"_wu{WARMUP_ITERS}_an{LE_ANNEAL}" if STAGE == 2 else "")
            + ("_initauthor" if INIT else "") + f"_seed{TRAIN_SEED}_it{ITERS}")

import os, sys, re, math, json, time, subprocess
import torch

GPU = torch.cuda.is_available()
GPU_NAME = torch.cuda.get_device_name(0) if GPU else "CPU"
THREADS = max(2, min(8, os.cpu_count() or 2))   # dataloader workers: runtime ที่ GPU แรงกว่ามักมี CPU มากกว่า
print("torch", torch.__version__, "| GPU:", GPU_NAME,
      f"({torch.cuda.get_device_properties(0).total_memory / 1e9:.0f} GB)" if GPU else "", "| CPU cores:", os.cpu_count())
if not GPU:
    print(">>> ยังเป็น CPU: Runtime -> Change runtime type -> T4 GPU แล้วรันใหม่ตั้งแต่เซลล์นี้")

ROOT = "/content"
REPO = f"{ROOT}/UDnet"
UIEB = f"{ROOT}/uieb"        # ภาพที่แตกแล้ว (ดิสก์ของ runtime อ่านเร็ว)
DATA = f"{ROOT}/data"        # โครงสร้างโฟลเดอร์ที่สคริปต์ใช้
RESULTS = f"{ROOT}/results"  # ภาพผลลัพธ์
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = "/content/drive/MyDrive/UDnet_work"   # ใช้โฟลเดอร์เดียวกันทั้งสอง notebook
else:
    WORK = f"{ROOT}/UDnet_work"
CACHE = f"{WORK}/downloads"
RUN_DIR = f"{WORK}/runs/{RUN_NAME}"
for d in (CACHE, RUN_DIR, UIEB, DATA, RESULTS):
    os.makedirs(d, exist_ok=True)
with open(f"{RUN_DIR}/env.txt", "a") as f:     # GPU ของแต่ละ session ใน run นี้ — ไว้เช็กว่าเทียบกันบน GPU แบบเดียวกัน
    f.write(f"{time.strftime('%Y-%m-%d %H:%M')}  {GPU_NAME}  torch {torch.__version__}  cpu {os.cpu_count()}\n")
print("run:", RUN_NAME, "\nไฟล์ถาวรเก็บที่:", WORK)

## §2 Clone repo + ข้อมูล

In [ ]:
NEED = ["TrainU.py", "InferenceU.py", "Evaluate.py", "colab_utils.py", "model_utils/udnet_u.py"]

if not os.path.isdir(f"{REPO}/.git"):
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO_URL, REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull", "-q"], check=True)
os.chdir(REPO)
sys.path.insert(0, REPO)
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)
missing = [f for f in NEED if not os.path.exists(f)]
assert not missing, f"repo ขาดไฟล์ {missing} — ต้อง push branch {BRANCH} ขึ้น GitHub ก่อน"

import importlib.util
if importlib.util.find_spec("gdown") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
import colab_utils as cu

cu.prepare_uieb(UIEB, CACHE)
AUTHOR_W = cu.author_weights(CACHE)
split = cu.make_split(UIEB, f"{WORK}/split_seed{SEED}.json", SEED, 800)
cu.build_data_dirs(UIEB, DATA, split, N_SUBSET, SEED)

EUVP = f"{ROOT}/euvp_good"
cu.prepare_clean_domain(CACHE, EUVP)
cu.build_clean_dir(EUVP, DATA, N_CLEAN or N_SUBSET, SEED)

## §3 Smoke test — UDnet จริง บนข้อมูลจริง

ต่อจาก §3 ของ demo (ซึ่งตรวจความถูกต้องของสูตรบนข้อมูลสังเคราะห์) เซลล์นี้ตรวจว่าทุกชิ้น**ต่อกับ UDnet ของ repo แล้วทำงาน**: รูปร่าง, ค่าจำกัด, gradient ไหลถึง generator ในทุกแบบของ lock, และจำนวนพารามิเตอร์ขณะอนุมาน

In [ ]:
import torch
from model_utils.data import get_training_set
from model_utils.udnet_u import (UDnetG, PatchDiscriminator, PhysicalLoss, LearnedPhysics, underwater_dcp,
                                 apply_phase_lock, lsgan_g_loss, load_repo_weights)
from TrainU import CleanFolder

dev = "cuda" if GPU else "cpu"
torch.manual_seed(0)
ds = get_training_set(f"{DATA}/train/", "image", "image", 256, False)
X = torch.stack([ds[i][0] for i in range(2)]).to(dev)
PSEUDO = torch.stack([ds[i][1] for i in range(2)]).to(dev)
Yc = torch.stack([CleanFolder(f"{DATA}/clean", 256)[i] for i in range(2)]).to(dev)

ok_all = True
def check(name, cond, detail=""):
    global ok_all
    ok_all &= bool(cond)
    print(("  PASS  " if cond else "  FAIL  ") + name + ("   " + detail if detail else ""))

G, D, P = UDnetG().to(dev), PatchDiscriminator().to(dev), PhysicalLoss().to(dev)
G.train()
out, L_m, L_s = G(X, PSEUDO)
check("1. UDnet train forward: shape ตรงกับ input", out.shape == X.shape, str(tuple(out.shape)))
check("   KL (L_m, L_s) เป็นจำนวนจำกัด", bool(torch.isfinite(L_m) and torch.isfinite(L_s)),
      f"L_m={L_m.item():.4f} L_s={L_s.item():.4f}")
out_u, _, _ = G(X, Yc)
check("   posterior รับภาพสะอาด (ขั้น 2) ได้", out_u.shape == X.shape)

for mode in ["none", "full", "low", "high"]:
    G.zero_grad()
    o, a, b = G(X, PSEUDO)
    J = apply_phase_lock(o, X, mode, LOCK_RADIUS)
    (J.mean() + a + b).backward()
    g = sum(p.grad.abs().sum().item() for p in G.parameters() if p.grad is not None)
    rng = mode == "none" or (J.min() >= 0 and J.max() <= 1)
    check(f"2. lock={mode:<5s}: gradient ไหลถึง UDnet และค่าอยู่ใน [0,1]", g > 0 and rng, f"|grad|={g:.3e}")

t, B_inf = underwater_dcp(X)
check("3. DCP บนภาพจริง: t ใน [0.1, 1], B_inf (B,3)", bool(t.min() >= 0.1 and t.max() <= 1 and B_inf.shape == (2, 3)),
      f"t {t.min().item():.2f}..{t.max().item():.2f}  B_inf {[round(v, 3) for v in B_inf[0].tolist()]}")
lp, _ = P(out.detach(), X, t, B_inf)
check("4. physical loss (dcp) เป็นจำนวนจำกัด", bool(torch.isfinite(lp)), f"{lp.item():.4f}")
LPn = LearnedPhysics(GB_SIGMA).to(dev)
G.zero_grad()
o, a, b = G(X, PSEUDO)
lt, parts = LPn(o, X)
lt.backward()
g = sum(p.grad.abs().sum().item() for p in G.parameters() if p.grad is not None)
check("   physical loss (learned): จำกัด และ gradient ไหลถึง UDnet", bool(torch.isfinite(lt)) and g > 0,
      " ".join(f"{k}={v.item():.4f}" for k, v in parts.items())
      + f"  | {sum(p.numel() for p in LPn.parameters()) / 1e6:.2f}M (ใช้เฉพาะตอนเทรน)")
d = D(out.detach())
check("5. discriminator: patch map", d.dim() == 4 and d.shape[1] == 1,
      f"{tuple(d.shape)}  {sum(p.numel() for p in D.parameters()) / 1e6:.2f}M (ใช้เฉพาะตอนเทรน)")

G.eval()
with torch.no_grad():
    e = G(X)
n_inf = sum(p.numel() for n, p in G.named_parameters()
            if not n.startswith(("decoder.po_", "decoder.compute_z_po", "decoder.colornet")))
check("6. inference forward (prior mean) shape ตรง", e.shape == X.shape,
      f"params ขณะอนุมาน {n_inf / 1e6:.3f}M (เท่า UDnet เดิม — เปเปอร์เขียน 2.6M แต่โค้ดจริง 1.40M รวม posterior)")
G2 = UDnetG()
load_repo_weights(G2, AUTHOR_W)
print("=" * 72)
print("ผลรวม:", "ผ่านทั้งหมด" if ok_all else ">>> มีข้อที่ไม่ผ่าน อย่าเพิ่งเทรน <<<")

## §4 ด่านตัดสินใจ — ขอบบนของ phase lock บน UIEB จริง

ถามว่า *ต่อให้โมเดลทาย amplitude ได้ดีที่สุดเท่าที่เป็นไปได้ ภายใต้ lock แต่ละแบบ จะได้ PSNR เทียบ reference สูงสุดเท่าไร*
(ความถี่ที่ถูกล็อก: ฉายสเปกตรัมของ reference ลงบนทิศเฟสของ input · ความถี่ที่ปล่อย: ใช้ของ reference ตรงๆ)

- `full` = demo ตั้งค่า `LOCK_RADIUS=None` (ล็อกทุกความถี่) · `low` = demo ตั้ง `LOCK_RADIUS` (ล็อกต่ำ-กลาง ปล่อยสูง)
- `high` = ทางเลือกที่เสนอเพิ่ม: ล็อกเฉพาะความถี่สูง (ขอบ) ปล่อยความถี่ต่ำ (ที่หมอกกับแสงฉากหลังอยู่)

**วิธีอ่าน:** ถ้าขอบบนของแบบไหน **ไม่สูงกว่า 24.61 dB (Wavenet ในตาราง 2) อย่างชัดเจน** แบบนั้นคือเพดานของวิธีเอง เทรนดีแค่ไหนก็ข้ามไม่ได้

บรรทัด **"เป้าที่ physical loss ดึงไปหา"** คือภาพที่ทำให้ physical loss เป็น 0 พอดี (t, B∞ จาก DCP, γ=1) — ถ้า PSNR ของมัน**ต่ำกว่าภาพดิบ**
แปลว่ายิ่งเพิ่มน้ำหนัก `LAM_PHYS` ผลยิ่งถูกดึงไปทางที่แย่กว่าไม่ทำอะไร ให้ใช้ λ เล็ก หรือออกแบบ loss ใหม่

In [ ]:
import numpy as np, cv2, csv
from model_utils.metrics import psnr as psnr_u8
from model_utils.udnet_u import best_under_phase_lock, underwater_dcp

def load(p):
    return cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB)
def t_(x):
    return torch.from_numpy(x).permute(2, 0, 1).float().div(255).to(dev)
def u8(x):
    return (x.permute(1, 2, 0).cpu().numpy() * 255).round().astype(np.uint8)

modes = [("full", None), ("low", LOCK_RADIUS), ("high", LOCK_RADIUS), ("high", 0.25)]
res = {"raw": []}
res.update({f"{m} r={r}": [] for m, r in modes})
res["phys"] = []
with torch.no_grad():
    for n in split["test"]:
        raw, ref = load(f"{DATA}/test/input/{n}"), load(f"{DATA}/test/reference/{n}")
        res["raw"].append(psnr_u8(raw, ref))
        for m, r in modes:
            res[f"{m} r={r}"].append(psnr_u8(u8(best_under_phase_lock(t_(ref), t_(raw), m, r or 0.5)), ref))
        # image that makes the physical loss exactly 0 (gamma = 1): J* = (I - B(1 - t)) / t
        X1 = t_(raw).unsqueeze(0)
        t_hat, B_hat = underwater_dcp(X1)
        J_star = ((X1 - B_hat.view(1, 3, 1, 1) * (1 - t_hat)) / t_hat).clamp(0, 1)[0]
        res["phys"].append(psnr_u8(u8(J_star), ref))

author = f"{WORK}/metrics_author.csv"
print(f"{'':<36s}{'mean PSNR':>10s}{'min':>8s}{'max':>8s}")
for k, v in res.items():
    label = {"raw": "ภาพดิบ", "phys": "เป้าของ physical loss แบบ dcp (γ=1)"}.get(k, "ขอบบน lock " + k)
    print(f"{label:<36s}{np.mean(v):>10.2f}{np.min(v):>8.2f}{np.max(v):>8.2f}")
worst = sorted(zip(res["full r=None"], split["test"]))[:5]
print("\nภาพที่ lock full มีเพดานต่ำสุด (ดูว่าเป็นฉากไกล/หมอกหนาหรือไม่):", [f"{n} {p:.1f}" for p, n in worst])
if os.path.exists(author):
    print(f"{'UDnet weight ผู้แต่ง (วัดจริง)':<36s}{np.mean([float(r['PSNR']) for r in csv.DictReader(open(author))]):>10.2f}")
print(f"{'Wavenet (ตาราง 2 เปเปอร์)':<36s}{24.61:>10.2f}")

## §5 เทรน UDnet-U

`TrainU.py` บันทึก `last.pth` ทุก 500 iteration ลง `UDnet_work/runs/<RUN_NAME>/` — **runtime หลุด:** รัน §1–§2 ใหม่ แล้วรันเซลล์นี้ซ้ำ จะต่อจากจุดเดิมพร้อม state ของ optimizer
log ทุกพจน์ (รวม L_m และ L_s แยกกันตาม checklist) อยู่ใน `loss_log.csv`

In [ ]:
args = ["TrainU.py", "--gpu_mode", str(GPU), "--data_dir", f"{DATA}/train/", "--data_train_dataset", "image",
        "--clean_dir", f"{DATA}/clean", "--stage", str(STAGE), "--iters", str(ITERS), "--batchSize", str(BATCH),
        "--threads", str(THREADS), "--seed", str(TRAIN_SEED), "--lam_e", str(LAM_E), "--beta_kl", str(BETA_KL),
        "--lam_adv", str(LAM_ADV), "--adv_warmup", str(ADV_WARMUP), "--lam_phys", str(LAM_PHYS),
        "--phys_mode", PHYS_MODE, "--gb_sigma", str(GB_SIGMA),
        "--lock", LOCK, "--lock_radius", str(LOCK_RADIUS), "--warmup_iters", str(WARMUP_ITERS),
        "--le_anneal", str(LE_ANNEAL), "--save_folder", f"{RUN_DIR}/"]
if INIT == "author":
    args += ["--init", AUTHOR_W]
if os.path.exists(f"{RUN_DIR}/final.pth"):
    print("run นี้เทรนเสร็จแล้ว:", RUN_DIR)
else:
    cu.run_logged(args, f"{RUN_DIR}/train_log.txt")
    print("เทรนเสร็จ")

## §6 ประเมิน + ตารางรวมกับเส้นฐาน

`InferenceU.py` ใส่ phase lock แบบเดียวกับตอนเทรนให้เอง (lock เป็นส่วนหนึ่งของผลลัพธ์ จึงต้องใช้ตอนอนุมานด้วย)

In [ ]:
cu.run(["InferenceU.py", "--model", f"{RUN_DIR}/final.pth", "--gpu_mode", str(GPU),
        "--input_dir", f"{DATA}/test/input", "--output_dir", f"{RESULTS}/{RUN_NAME}"])
cu.run(["Evaluate.py", "--pred_dir", f"{RESULTS}/{RUN_NAME}", "--gt_dir", f"{DATA}/test/reference",
        "--csv", f"{RUN_DIR}/metrics_test.csv"])
print()
cu.summarize(WORK)

## §6.1 (ทางเลือก) เทรนสั้นกว่านี้พอไหม

`TrainU.py` เก็บ `iter_10000.pth`, `iter_20000.pth`, ... ไว้ระหว่างเทรน เซลล์นี้วัดผลของแต่ละจุดบนชุดทดสอบ
ถ้า 20,000 ใกล้กับ 40,000 run ที่เหลือในแผนใช้ `ITERS = 20000` ได้ ประหยัดครึ่งหนึ่ง (ต้องใช้ค่าเดียวกันทุก run รวมทั้ง `ITER_BUDGET` ของเส้นฐาน)

In [ ]:
import glob
snaps = sorted((int(re.search(r"iter_(\d+)\.pth$", p).group(1)), p) for p in glob.glob(f"{RUN_DIR}/iter_*.pth"))
snaps.append((ITERS, f"{RUN_DIR}/final.pth"))

def infer(ckpt, out):
    cu.run(["InferenceU.py", "--model", ckpt, "--gpu_mode", str(GPU), "--input_dir", f"{DATA}/test/input",
            "--output_dir", out], quiet=True)

cu.convergence(snaps, infer, f"{RESULTS}/{RUN_NAME}_conv", f"{DATA}/test/reference", RUN_DIR)

## §7 กราฟ loss และภาพเปรียบเทียบ

หัวข้อกราฟเป็นภาษาอังกฤษ เพราะ matplotlib บน Colab ไม่มีฟอนต์ไทย

In [ ]:
import matplotlib.pyplot as plt
log = list(csv.DictReader(open(f"{RUN_DIR}/loss_log.csv")))
cols = [k for k in ["L_e", "L_m", "L_s", "adv", "D", "phys"] if any(r[k] for r in log)]
fig, ax = plt.subplots(1, len(cols), figsize=(3.2 * len(cols), 2.8))
for a, k in zip(np.atleast_1d(ax), cols):
    pts = [(int(r["iter"]), float(r[k])) for r in log if r[k]]
    a.plot(*zip(*pts), lw=1.2); a.set_title(k); a.set_xlabel("iter"); a.grid(alpha=.3)
plt.tight_layout(); plt.show()

names = split["test"][:4]
rows = [("input", f"{DATA}/test/input"), ("UDnet-U", f"{RESULTS}/{RUN_NAME}"), ("reference", f"{DATA}/test/reference")]
if os.path.isdir(f"{RESULTS}/author"):
    rows.insert(1, ("UDnet (author)", f"{RESULTS}/author"))
fig, ax = plt.subplots(len(rows), len(names), figsize=(3 * len(names), 2.4 * len(rows)))
for i, (title, folder) in enumerate(rows):
    for j, n in enumerate(names):
        stem = os.path.splitext(n)[0]
        path = next(f"{folder}/{f}" for f in os.listdir(folder) if os.path.splitext(f)[0] == stem)
        ax[i, j].imshow(load(path)); ax[i, j].axis("off")
        if j == 0:
            ax[i, j].set_title(title, loc="left", fontsize=10)
plt.tight_layout(); plt.show()